In [2]:
import pandas as pd

df = pd.read_csv('online_retail_dataset.csv')

In [3]:
df.head()

,customer_id,age,annual_income,spending_score,visits_last_month,avg_session_time,purchased
0,1,56,33498.470581,80,2,9.205672,0
1,2,46,81461.043180,99,2,15.262898,1
2,3,32,80940.183493,30,6,6.029399,1
3,4,60,54094.755346,74,5,5.628839,1
4,5,25,60337.701168,18,4,6.415331,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        2000 non-null   int64  
 1   age                2000 non-null   int64  
 2   annual_income      2000 non-null   float64
 3   spending_score     2000 non-null   int64  
 4   visits_last_month  2000 non-null   int64  
 5   avg_session_time   2000 non-null   float64
 6   purchased          2000 non-null   int64  
dtypes: float64(2), int64(5)
memory usage: 109.5 KB


In [5]:
df.isnull().sum()   

customer_id          0
age                  0
annual_income        0
spending_score       0
visits_last_month    0
avg_session_time     0
purchased            0
dtype: int64

In [7]:
def cap_outliers(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    
    return col.clip(lower, upper)

df["annual_income"] = cap_outliers(df["annual_income"])
df["avg_session_time"] = cap_outliers(df["avg_session_time"])


In [9]:
df["income_per_visit"] = df["annual_income"]/(df["visits_last_month"]+1)


In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(["customer_id","purchased"], axis=1)
y = df["purchased"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier()

lr.fit(X_train,y_train)
rf.fit(X_train,y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def evaluate(model):
    pred = model.predict(X_test)
    
    print("Accuracy:",accuracy_score(y_test,pred))
    print("Precision:",precision_score(y_test,pred))
    print("Recall:",recall_score(y_test,pred))
    print("F1:",f1_score(y_test,pred))
    print("-----------")

print("Logistic Regression")
evaluate(lr)

print("Random Forest")
evaluate(rf)


Logistic Regression
Accuracy: 0.6225
Precision: 0.6433333333333333
Recall: 0.8143459915611815
F1: 0.7188081936685289
-----------
Random Forest
Accuracy: 0.5925
Precision: 0.6390977443609023
Recall: 0.7172995780590717
F1: 0.6759443339960238
-----------


In [14]:
import pandas as pd

imp = pd.Series(rf.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False))


annual_income        0.219543
avg_session_time     0.189477
income_per_visit     0.183068
spending_score       0.178418
age                  0.152774
visits_last_month    0.076720
dtype: float64


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_scaled = LogisticRegression()
lr_scaled.fit(X_train_s,y_train)

print("Scaled Logistic Regression")
evaluate(lr_scaled)


Scaled Logistic Regression
Accuracy: 0.5925
Precision: 0.5925
Recall: 1.0
F1: 0.7441130298273155
-----------


c:\Users\Ved Dhanokar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [16]:
new_customer = pd.DataFrame({
    "age":[30],
    "annual_income":[75000],
    "spending_score":[80],
    "visits_last_month":[6],
    "avg_session_time":[12],
})


new_customer["income_per_visit"] = \
    new_customer["annual_income"]/(new_customer["visits_last_month"]+1)


prediction = rf.predict(new_customer)
prob = rf.predict_proba(new_customer)

print("Prediction:", prediction)
print("Purchase Probability:", prob[0][1])


Prediction: [1]
Purchase Probability: 0.76
